# 02. Limpieza y Enriquecimiento de Datos (Feature Engineering)

Una vez que entendieron los datos, vamos a prepararlos para que los algoritmos de Machine Learning puedan procesarlos correctamente. En la vida real, lo mejor es empaquetar esto automatizado en Scikit-Learn Pipelines o en scripts de Python (`src/features/build_features.py`). Aquí puedes experimentar con las rutinas de limpieza.

### Instrucciones Generales:
1. **Solvertar problema de calidad:**: Solucionar problema de calidad encontrados en el EDA: consistencia, sensibilidad, precision y completitud. Documenta cada decision tomada.
2. **Codificación Categórica:** El campo `ocean_proximity` es de texto. Conviértelo en variable numerica, ya que los algoritmos clasicos no entienen el texto. Documenta porque usaste codificacion Ordinal o Nominal.
3. **Enriquecimiento (Feature Engineering):** Como pudiste notar en tu análisis, `total_rooms` no significa mucho si hay muchos hogares en un distrito. Agrega nuevas métricas útiles, por ejemplo:
   - `rooms_per_household = total_rooms / households`
   - `bedrooms_per_room = total_bedrooms / total_rooms`
   - `population_per_household = population / households`
4. **Escalado de Variables:** Aplica un `StandardScaler` o `MinMaxScaler` para evitar que las variables numéricas grandes pesen más en algoritmos basados en distancias o gradientes.


In [7]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

In [3]:
# Cargar los datos de entrenamiento
train_path = "../data/interim/train_set.csv"
housing = pd.read_csv(train_path)

# Separar las variables predictoras (X) de la etiqueta a predecir (y)
housing_X = housing.drop("median_house_value", axis=1)
housing_y = housing["median_house_value"].copy()

# Solucionar completitud (Valores nulos en total_bedrooms)
# Creamos una copia solo con números para que el imputer no falle con el texto
housing_num = housing_X.select_dtypes(include=[np.number])

imputer = SimpleImputer(strategy="median")
# Calculamos la mediana y transformamos los datos
X_imputed = imputer.fit_transform(housing_num)

# Reconstruimos el DataFrame numérico limpio
housing_num_clean = pd.DataFrame(X_imputed, columns=housing_num.columns, index=housing_num.index)

housing_num_clean.isnull().sum()

longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
dtype: int64

In [ ]:


# 3. Codificación de la variable categórica
housing_cat = housing_X[["ocean_proximity"]]

# Usamos OneHotEncoder (crea una columna binaria 1/0 para cada categoría)
cat_encoder = OneHotEncoder(sparse_output=False)
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)

# Convertimos a DataFrame para visualizarlo fácilmente
categorias = cat_encoder.get_feature_names_out()
housing_cat_encoded = pd.DataFrame(housing_cat_1hot, columns=categorias, index=housing_X.index)

display(housing_cat_encoded.head())

,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,0.0,0.0,0.0,1.0,0.0
1,1.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0
3,0.0,1.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,1.0


In [5]:
# 4. Feature Engineering sobre nuestro DataFrame limpio
housing_extra = housing_num_clean.copy()

# Habitaciones por hogar
housing_extra["rooms_per_household"] = housing_extra["total_rooms"] / housing_extra["households"]
# Porcentaje de dormitorios respecto al total de habitaciones
housing_extra["bedrooms_per_room"] = housing_extra["total_bedrooms"] / housing_extra["total_rooms"]
# Personas por hogar
housing_extra["population_per_household"] = housing_extra["population"] / housing_extra["households"]

print("--- Nuevas características agregadas ---")
display(housing_extra[["rooms_per_household", "bedrooms_per_room", "population_per_household"]].head())

--- Nuevas características agregadas ---


,rooms_per_household,bedrooms_per_room,population_per_household
0,3.211799,0.335742,1.524178
1,5.504202,0.180153,1.865546
2,5.334975,0.200369,2.768473
3,5.351282,0.203881,2.365385
4,3.725256,0.277371,1.631399


In [9]:
from sklearn.preprocessing import StandardScaler

# 5. Escalar las variables numéricas (incluyendo las nuevas)
scaler = StandardScaler()
housing_scaled_array = scaler.fit_transform(housing_extra)

# Reconstruimos el DataFrame final escalado
housing_scaled = pd.DataFrame(housing_scaled_array, columns=housing_extra.columns, index=housing_extra.index)

# Finalmente, unimos la parte numérica escalada con la parte categórica codificada
housing_prepared = pd.concat([housing_scaled, housing_cat_encoded], axis=1)

display(housing_prepared.head())
housing_prepared.shape

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,rooms_per_household,bedrooms_per_room,population_per_household,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,-1.423037,1.013606,1.861119,0.311912,1.368167,0.137460,1.394812,-0.936491,-0.866027,1.846624,-0.330204,0.0,0.0,0.0,1.0,0.0
1,0.596394,-0.702103,0.907630,-0.308620,-0.435925,-0.693771,-0.373485,1.171942,0.024550,-0.508121,-0.253616,1.0,0.0,0.0,0.0,0.0
2,-1.203098,1.276119,0.351428,-0.712240,-0.760709,-0.788768,-0.775727,-0.759789,-0.041193,-0.202155,-0.051041,0.0,1.0,0.0,0.0,0.0
3,1.231216,-0.884924,-0.919891,0.702262,0.742306,0.383175,0.731375,-0.850281,-0.034858,-0.149006,-0.141475,0.0,1.0,0.0,0.0,0.0
4,0.711362,-0.875549,0.589800,0.790125,1.595753,0.444376,1.755263,-0.180365,-0.666554,0.963208,-0.306148,0.0,0.0,0.0,0.0,1.0


(16512, 16)

### Documentación de Decisiones (Feature Engineering)

1. Calidad y Completitud: Se detectaron valores faltantes en total_bedrooms. Se optó por utilizar una estrategia de imputación basada en la mediana (SimpleImputer) para evitar que los valores outliers de distritos con mansiones o complejos masivos distorsionaran el valor de reemplazo. 
2. Codificación Categórica: Para la variable ocean_proximity, se eligió una codificación Nominal utilizando OneHotEncoder. No se utilizó codificación Ordinal porque las categorías geográficas no poseen una relación matemática o jerárquica secuencial.
3. Escalado: Se aplicó StandardScaler (estandarización) sobre todas las variables numéricas para asegurar que los algoritmos de gradiente descendente y basados en distancias converjan de manera eficiente, evitando que variables con magnitudes muy altas dominen la función de costo sobre variables de menor escala